# Linear Regression Model From Scratch
Given our response vector $Y$, a predictor matrix $X$, and a coefficients vector $\hat{\beta}$, our model will approxiamte the response as a linear relation:
$$Y \approx X\hat{\beta}.$$
First we'll do this by the **OLS** (ordinary least squares) method by minimizing the **residual sum of squares**
$$RSS=\sum_{i=1}^{n}(y_{i}-\hat{y_{i}})^{2}=\sum_{i=1}^{n}(y_{i}-\hat{\beta_{0}}-\hat{\beta_{1}}x_{i 1}-\dots-\hat{\beta_{p}}x_{ip})^{2},$$
via the **normal equation**:
$$\theta = \hat{\beta} = (X^{T}X)^{-1}X^{T}Y,$$
wich gives us the best estimate for $\hat{\beta}$ given $X$ and $Y$.

In [1]:
# Import numpy, the main library for creating the model
import numpy as np

In [2]:
class LinearRegression:
    """A linear regression model with both a fit() and predict() methods.

    The model is made to work with quantitative predictors, and qualitative predictors 
    should be taken into account when creating the observations matrix.

    Attributes:
        coefficients(np.array(float)): One dimensional array with the model estimated coefficients.
        intercept(float): Intercept coefficient.
        fit_intercept(bool): Indicates if the model will have an intercept.
    """

    def __init__(self, intercept : bool = True):
        """Initializes the model with or without an intercept.

        Args:
            intercept(bool): Indicates if the model will have an intercept. Set to True by default.
        """
        self.coefficients = None
        self.intercept = None
        self.fit_intercept = intercept

    def fit(self, X : np.ndarray, y : np.ndarray) -> LinearRegression:
        """Fits the model given observations.

        Args:
            X(np.array(np.array(float))): Matrix containing the observations predictors.
            y(np.arrray(float)): Response vector for the observations.

        Returns:
            LinearRegression class with fitted coefficients.
        
        Raises:
            ValueError if the matrices are of the wrong sizes.
        """
        # Convert X and y to arrays in case they are not
        X = np.asarray(X)
        y = np.asarray(y)

        # Add bias term
        if self.fit_intercept:
            X = np.column_stack([np.ones(X.shape[0]), X])

        try:
            A : np.ndarray = X.T @ X 
            b : np.ndarray = X.T @ y
            # Solves the system Ax=b
            theta : np.ndarray = np.linalg.solve(A, b)

            if self.fit_intercept:
                self.intercept = theta[0]
                self.coefficients = theta[1:]
            else:
                self.intercept = 0
                self.coefficients = theta
            
            return self
        
        except ValueError:
            print("X must have (n,p) shape, and y must have shape n")
        

    def predict(self, X : np.ndarray) -> float | np.ndarray:
        """Gives a prediction given a set of data.

        Args:
            X(np.array(float)): Vector containing the data to make a prediction.

        Returns: 
            Prediction(float) for the vector X.
        """
        # Checks if the model is fitted
        if self.coefficients is None:
            raise ValueError("Model not fitted. Call fit() first.")

        X : np.ndarray = np.asarray(X)

        return X @ self.coefficients + self.intercept


In [3]:
# Synhetic observations
n : int = 4000   # number of observations
p : int = 10     # number of predictors
error : np.ndarray = np.random.normal(loc=0, scale=1, size=n)         # (nx1) error vector
X : np.ndarray = np.random.uniform(size = (n,p))                      # (nxp) observations
beta0 : float = 5                                                     # intercept
beta : np.ndarray = np.random.randint(low=3, high=20, size=p)         # (px1) random predictors
Y : np.ndarray = beta0 + (X @ beta) + error                           # (nx1) real results

In [4]:
beta

array([ 8, 17,  9,  9,  7, 13,  7,  7, 12,  5])

In [6]:
model = LinearRegression()

In [9]:
results = model.fit(X, Y)
print(f"Los coeficientes son: {results.intercept}, {results.coefficients}")

Los coeficientes son: 5.095357652351055, [ 8.0095734  17.08542841  8.96380575  8.94241628  6.93714141 12.97563382
  6.95894446  7.03729903 11.97192321  4.95235589]


In [10]:
X_predict : np.ndarray = np.random.normal(loc=0, scale=1, size=p)
X_predict

array([ 0.05970129, -0.69581813, -0.84225878, -1.05698882,  0.65654096,
       -0.70235105,  0.4969816 ,  0.29735141,  0.17308539, -0.41644797])

In [11]:
results.predict(X_predict)

np.float64(-22.3148375011574)

Now, let's compare the model's estimates for $\beta$ to scikit-learn's linear regression model.

In [17]:
# Import sklearn's linear regression model
from sklearn.linear_model import LinearRegression as LR
reg = LR().fit(X,Y)
reg.coef_

array([ 7.86701631, 16.96963117,  9.04107592, 19.00896594,  2.93903051,
       10.98406486, 15.00759998,  5.89706334, 19.00690318, 15.92270945])

In [16]:
for a, b in zip(reg.coef_, results.coefficients):
    print(a-b)

-1.412203687323199e-13
-1.5631940186722204e-13
1.829647544582258e-13
-4.369837824924616e-13
4.3876013933186186e-13
5.879741138414829e-13
3.3573144264664734e-13
1.900701818158268e-13
5.080380560684716e-13
-1.1191048088221578e-13


We can see that the coefficients estimated by sklearn's LinearRegression and the coefficients estimated by our own linear model match up to 13 decimal places.

#### Batch Gradient Descent Algorithm
Now, instead of estimating the coefficients using the **normal equation**, we'll do it via the batch gradient descent algorithm.
Let's assume our training data set consists of $n$ observations, each one with $p+1$ predictors (counting the slope term).
Given our cost/loss function $J(\theta)$, in this case the Mean Squared Error (MSE):
$$J(\theta)=\frac{1}{n}\sum_{i=1}^{n}(\hat{y}_{i}-y_{i})^{2}.$$
We want to compute $\nabla J(\theta)$ and use it to estimate the coefficients using gradient descent:
$$\theta_{k+1}=\theta_{k}-\alpha\nabla J(\theta_{k})\quad\alpha\text{ the learning rate}.$$
Let $X$ be our $n\times p+1$ matrix with our training observations, $y$ and $\hat{y}$ the $n\times 1$ vectors with the real values and the models predictions given $\theta$, respectively. 
Then we have that
$$\nabla J(\theta)=\frac{2}{n}X^{T}(\hat{y}-y),$$
and thus our algorithm should be of the form
$$\theta_{k+1}=\theta_{k}-\alpha\frac{2}{n}X^{T}(\hat{y}-y).$$

In [21]:
class BGD_LinearRegression:
    def __init__(self, intercept : bool = True):
        self.coefficients = None
        self.intercept = None
        self.fit_intercept = intercept
        
    def predict(self, X : np.ndarray) -> float:
            """Gives a prediction given a set of data.
    
            Args:
                X(np.array(float)): Vector containing the data to make a prediction.
    
            Returns: 
                Prediction(float) for the vector X.
            """
            # Checks if the model is fitted
            if self.coefficients is None:
                raise ValueError("Model not fitted. Call fit() first.")
    
            X : np.ndarray = np.asarray(X)
    
            return X @ self.coefficients + self.intercept
    
    def fit(self, X : np.ndarray, y : np.ndarray, learning_rate : float, iterations : int):
        # Prevent errors
        if iterations < 100:
            raise ValueError("Insuficcient number of iterations, please increase them")
        
        # Convert X and y to arrays in case they are not
        X = np.asarray(X)
        y = np.asarray(y)
        
        # Set learning rate and number of iterations
        alpha = learning_rate
        epochs = iterations
        
        # Add bias term
        if self.fit_intercept:
            X_fit = np.column_stack([np.ones(X.shape[0]), X])
        
        n, p = X_fit.shape
        
        # Initialize theta the coefficients vector
        theta = np.zeros(p)
        
        for _ in range(epochs):
            # Set the coefficients to the current estimate
            self.coefficients = theta[1:]
            self.intercept = theta[0]
            
            # Make a prediction using the estimates
            yhat = self.predict(X)
            
            # Compute the gradient
            gradient = (2 / n) * X_fit.T @ (yhat - y)
            
            # Update coefficients
            theta = theta - alpha*gradient
        
        # Set coefficients to the last estimate
        self.coefficients = theta[1:]
        self.intercept = theta[0]
        
        return self

In [22]:
bgd_model = BGD_LinearRegression()

In [28]:
lr = 0.1                   # Learning rate for BGD
epochs = 5000               # Number of epochs
bgd_results = bgd_model.fit(X, Y, lr, epochs)
print(f"Los coeficientes son: {bgd_results.intercept}, {bgd_results.coefficients}")

Los coeficientes son: 5.095357653059379, [ 8.0095734  17.08542841  8.96380575  8.94241628  6.93714141 12.97563382
  6.95894446  7.03729903 11.97192321  4.95235589]


In [24]:
print(f"Los coeficientes son: {results.intercept}, {results.coefficients}")

Los coeficientes son: 5.095357652351055, [ 8.0095734  17.08542841  8.96380575  8.94241628  6.93714141 12.97563382
  6.95894446  7.03729903 11.97192321  4.95235589]
